In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import time
from tqdm import tqdm
import seaborn as sns
import tempnet as tn
from scipy.linalg import eigh
import tempnet as tn
from scipy.sparse.linalg import expm
from expm import mfp_exp 

In [ ]:
def eigen_decomposition_step(A):
    d=np.sum(A, axis=1)
    isolated = d == 0
    A[isolated, np.where(isolated)[0]] = 1
    d[isolated] = 1
    d_inv_sqrt = 1.0 / np.sqrt(d)
    L_sym = np.eye(len(d)) - (d_inv_sqrt[:, None] * A * d_inv_sqrt[None, :])
    eigvals, Q = eigh(L_sym)
    Q_scaled = d_inv_sqrt[:, None] * Q          # D^{-1/2} Q
    Qt_scaled = Q.T * np.sqrt(d)[None, :]       # Q^T D^{1/2}
    return eigvals, Q_scaled, Qt_scaled

def expm_eig(eigvals, Q_scaled, Qt_scaled, scales):
    exp_terms = np.exp(-np.outer(scales, eigvals))
    return (Q_scaled * exp_terms[:, np.newaxis, :]) @ Qt_scaled



def analysis(event_table, label, n_scales=50):   
    tempo=tn.ContTempNetwork(events_table=event_table)
    tempo.compute_laplacian_matrices()
    scales = np.logspace(-6, 2, n_scales)

    tempo._compute_time_grid()
    times = tempo.time_grid.reset_index()['times'].unique()
    times.sort()

    n_steps = len(times) - 1

    times_eig = np.empty(n_steps)
    times_mul = np.empty(n_steps)
    times_expm = np.empty((n_steps, n_scales))
    times_paper = np.empty((n_steps, n_scales))

    for i in tqdm(range(n_steps)):
        try:
            A = tempo.compute_static_adjacency_matrix(
                start_time=times[i], end_time=times[i + 1]
            ).toarray()

            # Eigen decomposition
            t0 = time.perf_counter()
            eigvals, Q_scaled, Qt_scaled = eigen_decomposition_step(A)
            times_eig[i] = time.perf_counter() - t0

            # Vectorized multiplication
            t0 = time.perf_counter()
            exp_terms = np.exp(-np.outer(scales, eigvals))
            _ = (Q_scaled * exp_terms[:, np.newaxis, :]) @ Qt_scaled
            times_mul[i] = time.perf_counter() - t0

            # scipy expm
            L = tempo.laplacians[i]
            for j, s in enumerate(scales):
                t0 = time.perf_counter()
                _ = expm(-s * L)
                times_expm[i, j] = time.perf_counter() - t0

            # exp_paper
            for j, s in enumerate(scales):
                t0 = time.perf_counter()
                _ = mfp_exp(-s *L, err=1e-8, non_norm=0)
                times_paper[i, j] = time.perf_counter() - t0
        except Exception as e:
            print(e)


    total_eig = times_eig.sum()
    total_mul = times_mul.sum()
    total_eigen_method = total_eig + total_mul
    total_expm = times_expm.sum()
    total_paper = times_paper.sum()

    mean_expm_per_scale = times_expm.mean(axis=0)   # (n_scales,)
    mean_paper_per_scale = times_paper.mean(axis=0)  # (n_scales,)

    print(f"{'':=<70}")
    print(f"  Benchmark Report ({tempo.num_nodes} Nodes {n_steps} steps, {n_scales} scales, network: {label})")
    print(f"{'':=<70}")

    print(f"\n--- Eigen Method Breakdown (per slice) ---")
    print(f"{'Phase':<25} {'Mean (ms)':>10} {'Std (ms)':>10}")
    print(f"{'':-<45}")
    print(f"{'Eigen decomposition':<25} {times_eig.mean()*1e3:>10.2f} {times_eig.std()*1e3:>10.2f}")
    print(f"{'Multiplication (all scales)':<25} {times_mul.mean()*1e3:>10.2f} {times_mul.std()*1e3:>10.2f}")

    print(f"\n--- Total Time Comparison ---")
    print(f"{'Method':<25} {'Total (s)':>10} {'Speedup':>10}")
    print(f"{'':-<45}")
    print(f"{'scipy expm':<25} {total_expm:>10.3f} {'(base)':>10}")
    print(f"{'Eigen (decomp + mul)':<25} {total_eigen_method:>10.3f} {total_expm/total_eigen_method:>9.1f}x   ")
    print(f"{'exp_paper':<25} {total_paper:>10.3f} {total_expm/total_paper:>9.1f}x")

    print(f"\n--- Average Time per Scale (across slices) ---")
    print(f"{'Scale':<12} {'expm (ms)':>10} {'exp_paper (ms)':>15}")
    print(f"{'':-<37}")
    for j, s in enumerate(scales[::5]):
        print(f"{s:<12.2e} {mean_expm_per_scale[j]*1e3:>10.2f} {mean_paper_per_scale[j]*1e3:>15.2f}")

    print(f"{'':=<70}")

In [8]:
df = pd.read_csv("Data/synthetic.csv")
analysis(event_table=df, label='primary_school')

2026-05-18 21:24:00,572 - INFO - tempnet/temporal_network.py:1267 - PID:841767 - Computing Laplacians
2026-05-18 21:24:00,622 - INFO - tempnet/temporal_network.py:1326 - PID:841767 - 0 over 4
2026-05-18 21:24:00,624 - INFO - tempnet/temporal_network.py:1330 - PID:841767 - 0.01s
2026-05-18 21:24:00,793 - INFO - tempnet/temporal_network.py:1377 - PID:841767 - Finished in 0.17528223991394043
 25%|██▌       | 1/4 [00:00<00:01,  1.75it/s]/home/b/yasgar/.local/lib/python3.10/site-packages/scipy/sparse/linalg/_eigen/_svds.py:477: UserWarning: The problem size 3 minus the constraints size 0 is too small relative to the block size 1. Using a dense eigensolver instead of LOBPCG iterations.No output of the history of the iterations.
  _, eigvec = lobpcg(XH_X, X, tol=tol ** 2, maxiter=maxiter,
100%|██████████| 4/4 [00:02<00:00,  1.93it/s]

  Benchmark Report (3 Nodes 4 steps, 50 scales, network: primary_school)

--- Eigen Method Breakdown (per slice) ---
Phase                      Mean (ms)   Std (ms)
---------------------------------------------
Eigen decomposition            13.42      22.77
Multiplication (all scales)       0.85       1.36

--- Total Time Comparison ---
Method                     Total (s)    Speedup
---------------------------------------------
scipy expm                     0.749     (base)
Eigen (decomp + mul)           0.057      13.1x   
exp_paper                      1.247       0.6x

--- Average Time per Scale (across slices) ---
Scale         expm (ms)  exp_paper (ms)
-------------------------------------
1.00e-06           6.80           15.20
4.29e-05           4.30            3.05
1.84e-03           3.73            2.89
7.91e-02           3.58            2.95
3.39e+00           3.43            2.84


In [2]:
df = pd.read_csv("Data/primaryschool_clean.csv")
analysis(event_table=df, label='primary_school')

2026-05-18 23:03:16,932 - INFO - tempnet/temporal_network.py:1267 - PID:1209748 - Computing Laplacians
2026-05-18 23:03:17,256 - INFO - tempnet/temporal_network.py:1326 - PID:1209748 - 0 over 3101
2026-05-18 23:03:17,258 - INFO - tempnet/temporal_network.py:1330 - PID:1209748 - 0.02s
2026-05-18 23:04:01,911 - INFO - tempnet/temporal_network.py:1326 - PID:1209748 - 1000 over 3101
2026-05-18 23:04:01,913 - INFO - tempnet/temporal_network.py:1330 - PID:1209748 - 44.68s
2026-05-18 23:04:41,096 - INFO - tempnet/temporal_network.py:1326 - PID:1209748 - 2000 over 3101
2026-05-18 23:04:41,098 - INFO - tempnet/temporal_network.py:1330 - PID:1209748 - 83.86s
2026-05-18 23:05:23,689 - INFO - tempnet/temporal_network.py:1326 - PID:1209748 - 3000 over 3101
2026-05-18 23:05:23,691 - INFO - tempnet/temporal_network.py:1330 - PID:1209748 - 126.45s
2026-05-18 23:05:26,872 - INFO - tempnet/temporal_network.py:1377 - PID:1209748 - Finished in 129.63560104370117
 50%|█████     | 1555/3101 [1:16:31<1:04:42

cannot convert float infinity to integer


100%|██████████| 3101/3101 [2:30:14<00:00,  2.91s/it]  

  Benchmark Report (242 Nodes 3101 steps, 50 scales, network: primary_school)

--- Eigen Method Breakdown (per slice) ---
Phase                      Mean (ms)   Std (ms)
---------------------------------------------
Eigen decomposition           108.68     240.89
Multiplication (all scales)     167.64      46.17

--- Total Time Comparison ---
Method                     Total (s)    Speedup
---------------------------------------------
scipy expm                  6854.109     (base)
Eigen (decomp + mul)         856.842       8.0x   
exp_paper                   1285.815       5.3x

--- Average Time per Scale (across slices) ---
Scale         expm (ms)  exp_paper (ms)
-------------------------------------
1.00e-06          66.54            2.98
4.29e-05          56.15            2.58
1.84e-03          43.55            2.53
7.91e-02          42.79            2.51
3.39e+00          42.71            2.51


In [9]:
df = pd.read_csv("Data/mice_clean.csv")
df=df[df['starting_times']<3600]
analysis(event_table=df, label='Mice', n_scales=20)

2026-05-19 14:16:44,197 - INFO - tempnet/temporal_network.py:1267 - PID:4129243 - Computing Laplacians
2026-05-19 14:16:44,225 - INFO - tempnet/temporal_network.py:1326 - PID:4129243 - 0 over 1977
2026-05-19 14:16:44,227 - INFO - tempnet/temporal_network.py:1330 - PID:4129243 - 0.01s
2026-05-19 14:16:47,001 - INFO - tempnet/temporal_network.py:1326 - PID:4129243 - 1000 over 1977
2026-05-19 14:16:47,002 - INFO - tempnet/temporal_network.py:1330 - PID:4129243 - 2.78s
2026-05-19 14:16:50,730 - INFO - tempnet/temporal_network.py:1377 - PID:4129243 - Finished in 6.508569955825806
100%|██████████| 1977/1977 [48:10<00:00,  1.46s/it] 

  Benchmark Report (257 Nodes 1977 steps, 20 scales, network: Mice)

--- Eigen Method Breakdown (per slice) ---
Phase                      Mean (ms)   Std (ms)
---------------------------------------------
Eigen decomposition           126.81     235.01
Multiplication (all scales)     159.63      67.97

--- Total Time Comparison ---
Method                     Total (s)    Speedup
---------------------------------------------
scipy expm                  1964.886     (base)
Eigen (decomp + mul)         566.298       3.5x   
exp_paper                    348.163       5.6x

--- Average Time per Scale (across slices) ---
Scale         expm (ms)  exp_paper (ms)
-------------------------------------
1.00e-06          72.42            2.01
1.62e-02          57.66            1.84
